# 📊 Análisis Exploratorio de Datos — InfoRetail
**Proyecto:** EDA de ventas, productos, sucursales y clientes  
**Herramientas:** Python · Pandas · NumPy  
**Dataset:** Datos simulados de ventas

---
### Contenido del notebook
1. Importación de librerías
2. Generación del dataset simulado
3. Análisis Inicial de Datos (IDA)
4. Estadísticas Descriptivas
5. Análisis de Valores Nulos y Duplicados

---
## 1️⃣ Importación de librerías

In [ ]:
import numpy as np
import pandas as pd

# Mostrar todas las columnas en los DataFrames
pd.set_option('display.max_columns', None)
# Formato de números flotantes (2 decimales)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Librerías importadas correctamente')

---
## 2️⃣ Generación del dataset simulado

> 💡 **Nota:** En un proyecto real esta celda se reemplaza por:  
> `df = pd.read_csv('ventas.csv')` o `df = pd.read_excel('ventas.xlsx')`

El dataset representa registros de ventas de **InfoRetail** con las siguientes columnas:

| Columna | Descripción |
|---|---|
| `id_venta` | Identificador único de la transacción |
| `fecha` | Fecha y hora de la venta |
| `sucursal` | Sucursal donde se realizó la venta |
| `categoria` | Categoría del producto |
| `producto` | Nombre del producto |
| `cantidad` | Unidades vendidas |
| `precio_unitario` | Precio por unidad ($) |
| `descuento_pct` | Descuento aplicado (%) |
| `metodo_pago` | Forma de pago utilizada |
| `id_cliente` | Identificador del cliente |
| `edad_cliente` | Edad del cliente |
| `ingreso_bruto` | cantidad × precio_unitario |
| `ingreso_neto` | ingreso_bruto × (1 - descuento/100) |

In [ ]:
np.random.seed(42)  # Para reproducibilidad
N = 1000            # Número de registros

# Valores posibles para variables categóricas
categorias   = ['Electrónica', 'Ropa', 'Alimentos', 'Hogar', 'Deportes']
sucursales   = ['Buenos Aires', 'Córdoba', 'Rosario', 'Mendoza', 'Tucumán']
metodos_pago = ['Efectivo', 'Tarjeta de Crédito', 'Tarjeta de Débito', 'Transferencia']

# Construcción del DataFrame
df = pd.DataFrame({
    'id_venta'       : range(1, N + 1),
    'fecha'          : pd.date_range(start='2023-01-01', periods=N, freq='8h'),
    'sucursal'       : np.random.choice(sucursales, N),
    'categoria'      : np.random.choice(categorias, N),
    'producto'       : ['Producto_' + str(np.random.randint(1, 51)) for _ in range(N)],
    'cantidad'       : np.random.randint(1, 20, N),
    'precio_unitario': np.round(np.random.uniform(50, 5000, N), 2),
    'descuento_pct'  : np.random.choice([0, 5, 10, 15, 20], N, p=[0.5, 0.2, 0.15, 0.1, 0.05]),
    'metodo_pago'    : np.random.choice(metodos_pago, N),
    'id_cliente'     : np.random.randint(100, 600, N),
    'edad_cliente'   : np.random.randint(18, 70, N),
})

# Columnas calculadas
df['ingreso_bruto'] = (df['cantidad'] * df['precio_unitario']).round(2)
df['ingreso_neto']  = (df['ingreso_bruto'] * (1 - df['descuento_pct'] / 100)).round(2)

# Introducir valores nulos (~3%) para simular datos reales
for col in ['edad_cliente', 'descuento_pct', 'metodo_pago']:
    idx = np.random.choice(df.index, size=int(N * 0.03), replace=False)
    df.loc[idx, col] = np.nan

# Introducir filas duplicadas para práctica
filas_dup = df.sample(5, random_state=1)
df = pd.concat([df, filas_dup], ignore_index=True)

print(f'✅ Dataset generado: {df.shape[0]} filas × {df.shape[1]} columnas')

---
## 3️⃣ Análisis Inicial de Datos (IDA)

El **IDA** es el primer contacto con el dataset. Permite entender:
- La **estructura** del DataFrame (filas, columnas, tipos de dato)
- Los **primeros y últimos registros**
- Si existen problemas obvios de formato

> ⚠️ A diferencia del EDA, el IDA **no interpreta** los datos — solo examina su estructura.

In [ ]:
# Dimensiones del dataset
print(f'Filas   : {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')

In [ ]:
# Primeras 5 filas
df.head()

In [ ]:
# Últimas 5 filas
df.tail()

In [ ]:
# Muestra aleatoria — útil para ver registros del medio del dataset
df.sample(5, random_state=42)

In [ ]:
# Tipos de dato por columna y conteo de no-nulos
df.info()

In [ ]:
# Vista compacta de columnas con sus tipos y cantidad de nulos
pd.DataFrame({
    'columna' : df.columns,
    'dtype'   : df.dtypes.values,
    'no_nulos': df.notna().sum().values,
    'nulos'   : df.isna().sum().values
})

---
## 4️⃣ Estadísticas Descriptivas

Exploramos la **distribución y comportamiento** de los datos usando medidas de:
- **Tendencia central:** media, mediana
- **Dispersión:** desvío estándar
- **Posición:** percentiles, mínimo, máximo

In [ ]:
# Resumen estadístico de variables numéricas
df.describe().T

In [ ]:
# Estadísticas de variables categóricas
df.describe(include='object')

In [ ]:
# Conteo de valores únicos por columna categórica
for col in ['sucursal', 'categoria', 'metodo_pago']:
    print(f'\n--- {col.upper()} ---')
    print(df[col].value_counts())

In [ ]:
# Rango temporal del dataset
print(f'Fecha mínima : {df["fecha"].min()}')
print(f'Fecha máxima : {df["fecha"].max()}')
print(f'Período total: {(df["fecha"].max() - df["fecha"].min()).days} días')

In [ ]:
# KPIs generales de ingresos
print(f'Ingreso neto total : ${df["ingreso_neto"].sum():,.2f}')
print(f'Ticket promedio    : ${df["ingreso_neto"].mean():,.2f}')
print(f'Ticket mediano     : ${df["ingreso_neto"].median():,.2f}')
print(f'Desvío estándar    : ${df["ingreso_neto"].std():,.2f}')

In [ ]:
# Ingreso neto por sucursal
df.groupby('sucursal')['ingreso_neto'].agg(
    total='sum',
    promedio='mean',
    mediana='median',
    cant_ventas='count'
).sort_values('total', ascending=False).round(2)

In [ ]:
# Ingreso neto por categoría de producto
df.groupby('categoria')['ingreso_neto'].agg(
    total='sum',
    promedio='mean',
    cant_ventas='count'
).sort_values('total', ascending=False).round(2)

In [ ]:
# Top 10 productos por ingreso neto
df.groupby('producto')['ingreso_neto'].sum().sort_values(ascending=False).head(10).round(2)

In [ ]:
# Tabla cruzada: cantidad de ventas por sucursal y categoría
pd.crosstab(df['sucursal'], df['categoria'], margins=True)

---
## 5️⃣ Análisis de Valores Nulos y Duplicados

Antes de cualquier análisis complejo es fundamental:
- Cuantificar cuántos valores faltan y en qué columnas
- Decidir si **imputar**, **eliminar** o **mantener** los nulos
- Detectar y eliminar registros **duplicados**

In [ ]:
# Resumen de nulos: cantidad y porcentaje
nulos = pd.DataFrame({
    'nulos'     : df.isnull().sum(),
    'porcentaje': (df.isnull().sum() / len(df) * 100).round(2)
})

print('=== Resumen de valores nulos ===')
print(nulos[nulos['nulos'] > 0].sort_values('porcentaje', ascending=False))
print(f'\nTotal de celdas nulas: {df.isnull().sum().sum()}')

In [ ]:
# Filas que tienen al menos un valor nulo
filas_con_nulos = df[df.isnull().any(axis=1)]
print(f'Filas con al menos un nulo: {len(filas_con_nulos)} ({len(filas_con_nulos)/len(df)*100:.1f}%)')
filas_con_nulos.head()

In [ ]:
# ── TRATAMIENTO DE NULOS ──────────────────────────────────────
df_limpio = df.copy()  # Trabajar sobre copia para preservar el original

# edad_cliente (numérica) → imputar con la mediana
mediana_edad = df_limpio['edad_cliente'].median()
df_limpio['edad_cliente'] = df_limpio['edad_cliente'].fillna(mediana_edad)

# descuento_pct (numérica) → imputar con 0 (sin descuento si no hay dato)
df_limpio['descuento_pct'] = df_limpio['descuento_pct'].fillna(0)

# metodo_pago (categórica) → imputar con la moda
moda_pago = df_limpio['metodo_pago'].mode()[0]
df_limpio['metodo_pago'] = df_limpio['metodo_pago'].fillna(moda_pago)

print(f'Nulos restantes tras imputación: {df_limpio.isnull().sum().sum()}')

In [ ]:
# ── DUPLICADOS ────────────────────────────────────────────────
n_duplicados = df_limpio.duplicated().sum()
print(f'Filas duplicadas encontradas: {n_duplicados}')

# Ver las filas duplicadas
df_limpio[df_limpio.duplicated()]

In [ ]:
# Eliminar duplicados — mantener la primera ocurrencia
df_limpio = df_limpio.drop_duplicates(keep='first').reset_index(drop=True)

print(f'Filas antes de limpiar : {len(df)}')
print(f'Filas después de limpiar: {len(df_limpio)}')
print(f'Filas eliminadas        : {len(df) - len(df_limpio)}')

In [ ]:
# ── VALIDACIÓN FINAL ──────────────────────────────────────────
print('=== Dataset limpio — resumen final ===')
print(f'Filas        : {df_limpio.shape[0]}')
print(f'Columnas     : {df_limpio.shape[1]}')
print(f'Nulos totales: {df_limpio.isnull().sum().sum()}')
print(f'Duplicados   : {df_limpio.duplicated().sum()}')
print('\n✅ Dataset listo para análisis avanzado')

---
## 📝 Conclusiones

| Aspecto | Resultado |
|---|---|
| Dimensiones originales | 1.005 filas × 13 columnas |
| Columnas con nulos | `edad_cliente`, `descuento_pct`, `metodo_pago` (~3% c/u) |
| Tratamiento de nulos | Mediana (numérico), moda (categórico), 0 (descuento) |
| Duplicados eliminados | 5 filas |
| Dataset final | 1.000 filas × 13 columnas — listo para análisis |

> **Próximos pasos sugeridos:** visualizaciones exploratorias (histogramas, boxplots, correlación) y análisis por segmento de cliente o período temporal.